# 00 — First principles of proposal automation

> **Applied Labs** · **Domain**: PE · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/11_proposal_rfp_automation/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and NLP familiarity (embeddings, similarity)
- Understanding of text processing and structured data

**What you will learn**:
- Why RFP response is a *knowledge reuse + context adaptation* problem
- Why keyword search misses semantically equivalent past answers
- How context (industry, client size, requirements) determines the right adaptation
- Why cross-section consistency is critical and how to detect contradictions
- How compliance matrices translate requirements into structured evidence
- The $3 B+ proposal management market and the 20-40 hr pain point

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "sentence-transformers>=2.2.0" "numpy>=1.24.0"

import numpy as np
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import defaultdict
import re, textwrap, json

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — What is an RFP response?

A **Request for Proposal (RFP)** is a structured procurement document that
organizations issue to solicit bids from vendors. The responding company must
produce a **proposal** that addresses every question and requirement.

A typical RFP response contains:
- **Direct answers** to 10–200 specific questions
- A **compliance matrix** mapping each requirement to a capability
- **Pricing and SLA commitments**
- **References** and case studies
- An **executive summary** tailored to the prospect

The core challenge: each proposal takes **20–40 hours** of expert time, yet
**50%+ of content is reusable** across proposals. Let's define the problem
formally and build a sample RFP.

In [ ]:
@dataclass
class RFPQuestion:
    """A single question or requirement from an RFP."""
    id: str
    text: str
    category: str  # security, compliance, technical, pricing, support
    required: bool = True

@dataclass
class RFPDocument:
    """A complete RFP with metadata and questions."""
    title: str
    issuer: str
    industry: str
    company_size: str  # startup, mid-market, enterprise
    questions: List[RFPQuestion] = field(default_factory=list)

# Create a sample 10-question RFP
sample_rfp = RFPDocument(
    title="Cloud Platform Vendor Selection",
    issuer="Midwest Healthcare Group",
    industry="healthcare",
    company_size="enterprise",
    questions=[
        RFPQuestion("Q1", "Describe your data encryption approach for data at rest and in transit.", "security"),
        RFPQuestion("Q2", "What compliance certifications does your platform hold?", "compliance"),
        RFPQuestion("Q3", "Describe your disaster recovery and business continuity plan.", "technical"),
        RFPQuestion("Q4", "What is your guaranteed uptime SLA?", "technical"),
        RFPQuestion("Q5", "How do you handle HIPAA-protected health information?", "compliance"),
        RFPQuestion("Q6", "Describe your pricing model and any volume discounts.", "pricing"),
        RFPQuestion("Q7", "What support tiers do you offer and their response times?", "support"),
        RFPQuestion("Q8", "Describe your API integration capabilities.", "technical"),
        RFPQuestion("Q9", "How do you manage access control and user authentication?", "security"),
        RFPQuestion("Q10", "Provide at least two customer references in healthcare.", "references", required=False),
    ]
)

print(f"RFP: {sample_rfp.title}")
print(f"Issuer: {sample_rfp.issuer} ({sample_rfp.industry}, {sample_rfp.company_size})")
print(f"Questions: {len(sample_rfp.questions)}\n")
for q in sample_rfp.questions:
    req_tag = "REQUIRED" if q.required else "OPTIONAL"
    print(f"  [{q.id}] ({q.category:>10}, {req_tag:>8}) {q.text}")

## Section 2 — The knowledge reuse problem

Studies show that **50–70% of RFP answers** are reusable across proposals.
The challenge is *finding* them. Organizations store past answers in wikis,
shared drives, and CRM notes — retrieval is painful.

Let's build a **keyword-based retriever** and measure how many relevant past
answers it finds vs. misses. We'll see that keyword overlap fails when
questions are phrased differently but ask the same thing.

In [ ]:
# Past Q&A knowledge base (simplified)
past_answers: List[Dict[str, str]] = [
    {"question": "How do you encrypt customer data?",
     "answer": "We use AES-256 for data at rest and TLS 1.3 for data in transit.",
     "category": "security"},
    {"question": "What security certifications do you maintain?",
     "answer": "SOC 2 Type II, ISO 27001, HITRUST CSF, and FedRAMP Moderate.",
     "category": "compliance"},
    {"question": "Describe your backup and recovery procedures.",
     "answer": "Daily automated backups with 15-min RPO and 1-hr RTO. Geo-redundant.",
     "category": "technical"},
    {"question": "What uptime do you guarantee?",
     "answer": "99.9% uptime SLA backed by service credits.",
     "category": "technical"},
    {"question": "Are you HIPAA compliant?",
     "answer": "Yes. We sign BAAs, encrypt PHI, maintain access logs, and conduct annual audits.",
     "category": "compliance"},
    {"question": "How is your product priced?",
     "answer": "Per-seat monthly pricing with volume discounts starting at 100 users.",
     "category": "pricing"},
    {"question": "What support options are available?",
     "answer": "Standard (email, 24-hr), Premium (phone, 4-hr), Enterprise (dedicated CSM, 1-hr).",
     "category": "support"},
    {"question": "Do you offer REST APIs?",
     "answer": "Full REST API with OpenAPI spec, webhooks, and SDKs for Python/JS/Java.",
     "category": "technical"},
    {"question": "How is user access managed?",
     "answer": "RBAC with SSO (SAML 2.0, OIDC), MFA, and granular permission sets.",
     "category": "security"},
    {"question": "Can you provide healthcare customer references?",
     "answer": "Regional Health System (500-bed), CareFirst Clinic Network, MedTech Solutions.",
     "category": "references"},
]

print(f"Past answer knowledge base: {len(past_answers)} entries")
for i, qa in enumerate(past_answers):
    print(f"  [{i}] {qa['question'][:65]}...")

In [ ]:
def keyword_search(query: str, knowledge_base: List[Dict[str, str]],
                   top_k: int = 3) -> List[Tuple[int, float, str]]:
    """Simple keyword overlap retriever using Jaccard similarity.

    Returns list of (index, score, question) tuples sorted by score.
    """
    query_tokens: Set[str] = set(re.findall(r"\w+", query.lower()))
    results: List[Tuple[int, float, str]] = []
    for idx, qa in enumerate(knowledge_base):
        doc_tokens: Set[str] = set(re.findall(r"\w+", qa["question"].lower()))
        if len(query_tokens | doc_tokens) == 0:
            continue
        jaccard: float = len(query_tokens & doc_tokens) / len(query_tokens | doc_tokens)
        results.append((idx, jaccard, qa["question"]))
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

# Test keyword search on each RFP question
print("Keyword retrieval results:\n")
hits = 0
total = len(sample_rfp.questions)
for q in sample_rfp.questions:
    matches = keyword_search(q.text, past_answers, top_k=1)
    best_idx, best_score, best_q = matches[0] if matches else (-1, 0.0, "N/A")
    status = "✓ HIT" if best_score > 0.15 else "✗ MISS"
    if best_score > 0.15:
        hits += 1
    print(f"  {q.id}: score={best_score:.3f} {status}")
    print(f"    Query:    {q.text[:70]}")
    print(f"    Retrieved: {best_q[:70]}\n")

recall: float = hits / total
print(f"Keyword recall: {hits}/{total} = {recall:.1%}")
print("→ Keyword search misses semantically similar but differently worded questions.")

## Section 3 — Context adaptation

Even when a past answer is found, it can't be used verbatim. The same
question — *"Describe your security practices"* — requires different
emphasis depending on:

- **Industry**: Healthcare → HIPAA, PHI. Fintech → PCI-DSS, SOX.
- **Company size**: Startup → simplicity. Enterprise → governance.
- **Specific requirements**: The RFP may mandate specific certifications.

Let's build examples showing how a generic answer must be adapted.

In [ ]:
@dataclass
class AdaptationContext:
    """Context that controls how a past answer should be adapted."""
    industry: str
    company_size: str
    specific_requirements: List[str] = field(default_factory=list)
    tone: str = "formal"  # formal, conversational, technical

def demonstrate_adaptation_need(base_answer: str,
                                contexts: List[AdaptationContext]) -> None:
    """Show how the same base answer must change for different contexts."""
    print(f"Base answer:\n  {base_answer}\n")
    for ctx in contexts:
        gaps: List[str] = []
        if ctx.industry == "healthcare":
            if "HIPAA" not in base_answer:
                gaps.append("Must mention HIPAA compliance")
            if "PHI" not in base_answer:
                gaps.append("Must address PHI handling")
        elif ctx.industry == "fintech":
            if "PCI" not in base_answer:
                gaps.append("Must mention PCI-DSS")
            if "SOX" not in base_answer:
                gaps.append("Must address SOX requirements")
        if ctx.company_size == "enterprise":
            gaps.append("Add governance and audit details")
        elif ctx.company_size == "startup":
            gaps.append("Simplify; emphasize quick setup")
        for req in ctx.specific_requirements:
            if req.lower() not in base_answer.lower():
                gaps.append(f"Missing required mention: {req}")

        print(f"Context: {ctx.industry}, {ctx.company_size}")
        print(f"  Specific reqs: {ctx.specific_requirements}")
        print(f"  Adaptation gaps ({len(gaps)}):")
        for g in gaps:
            print(f"    → {g}")
        print()

# Demonstrate
base: str = "We use AES-256 encryption for data at rest and TLS 1.3 for data in transit."

contexts: List[AdaptationContext] = [
    AdaptationContext("healthcare", "enterprise", ["HIPAA", "BAA", "PHI encryption"]),
    AdaptationContext("fintech", "enterprise", ["PCI-DSS Level 1", "SOX compliance"]),
    AdaptationContext("education", "startup", ["FERPA"]),
]

demonstrate_adaptation_need(base, contexts)
print("→ Same base answer requires 2-5 adaptations per context.")

## Section 4 — Cross-section consistency

When 3–5 subject-matter experts each write sections independently, factual
claims can **contradict** each other. If section 3 promises "99.9% uptime"
but section 7 claims "99.99% availability," the proposal loses credibility.

Let's build a **consistency checker** concept that extracts factual claims
and detects contradictions.

In [ ]:
@dataclass
class FactualClaim:
    """A factual claim extracted from a proposal answer."""
    source_section: str
    claim_type: str   # uptime, response_time, certification, pricing
    value: str
    raw_text: str

def extract_claims(section: str, text: str) -> List[FactualClaim]:
    """Extract structured factual claims from answer text.

    Uses regex patterns to find quantitative and named claims.
    """
    claims: List[FactualClaim] = []
    # Uptime claims
    for m in re.finditer(r"(\d{2,3}\.\d+)\s*%\s*(uptime|availability)", text, re.I):
        claims.append(FactualClaim(section, "uptime", m.group(1), m.group(0)))
    # Response time claims
    for m in re.finditer(r"(\d+)\s*-?\s*(hour|hr|minute|min)\s*(response|RTO|RPO)", text, re.I):
        claims.append(FactualClaim(section, "response_time", f"{m.group(1)} {m.group(2)}", m.group(0)))
    # Certification claims
    certs = re.findall(r"(SOC\s*2|ISO\s*27001|HITRUST|FedRAMP|HIPAA|PCI-DSS)", text, re.I)
    for c in certs:
        claims.append(FactualClaim(section, "certification", c.upper(), c))
    return claims

def check_consistency(claims: List[FactualClaim]) -> List[Dict[str, str]]:
    """Detect contradictions among factual claims.

    Groups claims by type and flags differing values.
    """
    grouped: Dict[str, List[FactualClaim]] = defaultdict(list)
    for c in claims:
        grouped[c.claim_type].append(c)

    contradictions: List[Dict[str, str]] = []
    for claim_type, group in grouped.items():
        values = set(c.value for c in group)
        if len(values) > 1:
            contradictions.append({
                "type": claim_type,
                "values": list(values),
                "sections": [c.source_section for c in group],
                "details": [f"{c.source_section}: \"{c.raw_text}\"" for c in group],
            })
    return contradictions

# Simulate inconsistent proposal sections
section_answers: Dict[str, str] = {
    "Q3_DR": "Our platform guarantees 99.9% uptime with 1 hour RTO and 15 minute RPO.",
    "Q4_SLA": "We offer 99.99% availability with financially-backed SLA credits.",
    "Q2_Compliance": "We hold SOC 2 Type II, ISO 27001, and HITRUST certifications.",
    "Q5_HIPAA": "HIPAA compliant with SOC 2 and ISO 27001. We maintain 99.9% availability.",
}

all_claims: List[FactualClaim] = []
for section, text in section_answers.items():
    claims = extract_claims(section, text)
    all_claims.extend(claims)

print(f"Extracted {len(all_claims)} factual claims:\n")
for c in all_claims:
    print(f"  [{c.source_section}] {c.claim_type}: {c.value}")

contradictions = check_consistency(all_claims)
print(f"\n⚠ Found {len(contradictions)} contradiction(s):\n")
for con in contradictions:
    print(f"  Type: {con['type']}")
    print(f"  Conflicting values: {con['values']}")
    for d in con["details"]:
        print(f"    • {d}")
    print()

## Section 5 — The compliance matrix

Most RFPs include **must-have requirements**. The response must include a
**compliance matrix** — a structured table mapping each requirement to:

| Requirement | Capability | Evidence | Status |
|---|---|---|---|
| HIPAA compliance | BAA, PHI encryption | HITRUST cert | ✅ Fully compliant |

Let's build a compliance matrix generator.

In [ ]:
@dataclass
class ComplianceEntry:
    """A single row in a compliance matrix."""
    requirement: str
    capability: str
    evidence: str
    status: str  # fully_compliant, partially_compliant, not_compliant, planned

def generate_compliance_matrix(
    requirements: List[str],
    capabilities: Dict[str, Dict[str, str]]
) -> List[ComplianceEntry]:
    """Generate compliance matrix by matching requirements to capabilities.

    Args:
        requirements: List of RFP requirement strings.
        capabilities: Dict mapping keyword → {capability, evidence, status}.

    Returns:
        List of ComplianceEntry objects.
    """
    matrix: List[ComplianceEntry] = []
    for req in requirements:
        matched = False
        req_lower = req.lower()
        for keyword, cap_info in capabilities.items():
            if keyword.lower() in req_lower:
                matrix.append(ComplianceEntry(
                    requirement=req,
                    capability=cap_info["capability"],
                    evidence=cap_info["evidence"],
                    status=cap_info["status"],
                ))
                matched = True
                break
        if not matched:
            matrix.append(ComplianceEntry(req, "Under review", "N/A", "not_compliant"))
    return matrix

# Our capabilities lookup
capabilities: Dict[str, Dict[str, str]] = {
    "encryption": {"capability": "AES-256 at rest, TLS 1.3 in transit",
                   "evidence": "SOC 2 audit report §4.2", "status": "fully_compliant"},
    "hipaa": {"capability": "BAA, PHI encryption, access logging",
              "evidence": "HITRUST CSF certification", "status": "fully_compliant"},
    "uptime": {"capability": "99.9% uptime SLA with credits",
               "evidence": "Status page history, SLA document", "status": "fully_compliant"},
    "disaster recovery": {"capability": "Geo-redundant DR, 1-hr RTO",
                          "evidence": "DR test reports (quarterly)", "status": "fully_compliant"},
    "sso": {"capability": "SAML 2.0, OIDC, MFA",
            "evidence": "Integration documentation", "status": "fully_compliant"},
    "api": {"capability": "REST API with OpenAPI spec",
            "evidence": "API documentation portal", "status": "fully_compliant"},
    "fedramp": {"capability": "FedRAMP Moderate in progress",
                "evidence": "3PAO assessment scheduled", "status": "partially_compliant"},
}

rfp_requirements: List[str] = [
    "Vendor must provide encryption for data at rest and in transit",
    "Vendor must be HIPAA compliant and sign a BAA",
    "Vendor must guarantee 99.9% uptime or better",
    "Vendor must support SSO integration via SAML",
    "Vendor must provide API access for integration",
    "Vendor must have FedRAMP authorization",
    "Vendor must support real-time audit logging",
]

matrix = generate_compliance_matrix(rfp_requirements, capabilities)

STATUS_ICONS: Dict[str, str] = {
    "fully_compliant": "✅ Fully",
    "partially_compliant": "🟡 Partial",
    "not_compliant": "❌ No",
    "planned": "🔵 Planned",
}

print("Compliance Matrix\n" + "=" * 90)
for entry in matrix:
    icon = STATUS_ICONS.get(entry.status, "?")
    print(f"\n  Requirement: {entry.requirement}")
    print(f"  Capability:  {entry.capability}")
    print(f"  Evidence:    {entry.evidence}")
    print(f"  Status:      {icon}")

fully = sum(1 for e in matrix if e.status == "fully_compliant")
print(f"\nCompliance score: {fully}/{len(matrix)} fully compliant ({fully/len(matrix):.0%})")

## Section 6 — Market analysis

Proposal management is a **$3 B+** market driven by:

| Metric | Value |
|---|---|
| Market size | $3 B+ and growing 15% CAGR |
| Time per RFP | 20–40 hours of expert time |
| Content reuse | 50–70% of answers reusable |
| Win rate impact | Timely, high-quality proposals → 30% higher win rates |
| Key players | RFPIO/Responsive ($100 M+ ARR), Loopio, Qvidian |

The AI opportunity: **reduce 20–40 hrs to 2–4 hrs** while improving quality
and consistency. Companies responding to 100+ RFPs/year save **$500 K–$2 M**
annually.

In [ ]:
def calculate_roi(rfps_per_year: int, hours_per_rfp: float,
                  hourly_rate: float, automation_reduction: float) -> Dict[str, float]:
    """Calculate ROI of RFP automation.

    Args:
        rfps_per_year: Number of RFPs responded to annually.
        hours_per_rfp: Average hours per RFP response.
        hourly_rate: Fully loaded cost per expert hour.
        automation_reduction: Fraction of time saved (0.0 to 1.0).

    Returns:
        Dict with financial metrics.
    """
    current_cost: float = rfps_per_year * hours_per_rfp * hourly_rate
    new_hours: float = hours_per_rfp * (1 - automation_reduction)
    new_cost: float = rfps_per_year * new_hours * hourly_rate
    savings: float = current_cost - new_cost
    return {
        "rfps_per_year": rfps_per_year,
        "current_hours_total": rfps_per_year * hours_per_rfp,
        "current_cost": current_cost,
        "new_hours_per_rfp": new_hours,
        "new_cost": new_cost,
        "annual_savings": savings,
        "roi_percent": (savings / (new_cost + 50_000)) * 100,  # $50K platform cost
    }

# Scenarios
scenarios = [
    ("Small team",     50, 30, 150, 0.80),
    ("Mid-market",    150, 35, 175, 0.85),
    ("Enterprise",    400, 40, 200, 0.90),
]

print("RFP Automation ROI Analysis")
print("=" * 70)
for name, rfps, hrs, rate, reduction in scenarios:
    roi = calculate_roi(rfps, hrs, rate, reduction)
    print(f"\n  {name} ({rfps} RFPs/yr):")
    print(f"    Current cost:   ${roi['current_cost']:>12,.0f}  ({roi['current_hours_total']:.0f} hrs)")
    print(f"    Automated cost: ${roi['new_cost']:>12,.0f}  ({roi['new_hours_per_rfp']:.0f} hrs/RFP)")
    print(f"    Annual savings: ${roi['annual_savings']:>12,.0f}")
    print(f"    ROI:            {roi['roi_percent']:>11.0f}%")

## Exercises

1. **Improve the keyword retriever** — Replace Jaccard similarity with
   TF-IDF + cosine similarity. Measure whether recall improves on the 10
   sample RFP questions. Report precision and recall at k=3.

2. **Build an adaptation scorer** — Given a base answer and a target context,
   write a function that scores how many required adaptations the answer
   already addresses (0–100%). Test with at least 3 industry contexts.

3. **Extend the consistency checker** — Add detection for pricing
   contradictions (e.g., different per-seat costs in different sections).
   Create test cases with 3 intentional pricing inconsistencies.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | RFP response is a knowledge reuse + context adaptation problem |
| 2 | 50–70% of proposal content is reusable but hard to find with keywords |
| 3 | Keyword search achieves low recall because questions vary in phrasing |
| 4 | Context (industry, size, requirements) determines how answers must adapt |
| 5 | Cross-section consistency failures destroy proposal credibility |
| 6 | Compliance matrices require structured requirement → capability → evidence mapping |
| 7 | The $3 B+ market opportunity saves $500 K–$2 M/year for high-volume responders |

## What's Next

In **01 — Architecture**, we design the end-to-end RFP automation pipeline:
RFP ingestion, question extraction, RAG-based answer retrieval, context
adaptation, consistency checking, and proposal assembly.

---

## References

1. RFPIO/Responsive, *State of Strategic Response Management*, 2023 — <https://www.responsive.io/>
2. Loopio, *2023 RFP Response Trends*, 2023 — <https://loopio.com/>
3. Association of Proposal Management Professionals (APMP) — <https://www.apmp.org/>
4. Gartner, *Market Guide for Proposal Management Solutions*, 2023 — <https://www.gartner.com/>